# STAGE 3 step2.3：嘴巴开合三分类（按外观重新标注）HOG + SVM

替换原来的六类嘴型（closed/a/i/u/e/o）。原六类是按**发音**标注的，很多外观一样的图片标签不同，模型学不出来。
这里改成按**外观**分三类：

| 类别 | 外观 | 驱动 VRM |
|---|---|---|
| closed | 双唇闭合 | 嘴巴开合 0 |
| small | 微张，能看到少量牙齿或口腔 | 嘴巴开合 0.4 |
| wide | 大张，能看到大块口腔或舌头 | 嘴巴开合 1.0 |

参数用**留一人交叉验证**选择：每次留一个人当测试，其余人训练，所有人轮一遍后算平均 macro-F1。
这样不会因为验证集只有一个人而选出偶然的参数。

In [2]:
# OpenCV 版本检查：HOGDescriptor 只在 4.x 里有
from importlib.metadata import version, PackageNotFoundError
import subprocess, sys
try:
    cv_ver = version("opencv-python")
except PackageNotFoundError:
    cv_ver = None
print("opencv-python:", cv_ver)
if cv_ver is None or cv_ver.startswith("5"):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "opencv-python<5"])
    print("已安装 OpenCV 4.x，请 Restart Kernel 后再从头运行")

opencv-python: 4.10.0.84


In [3]:
from pathlib import Path
from collections import Counter
from datetime import datetime
import csv, json, pickle, uuid, itertools

import cv2
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix)
print(cv2.__version__)

# 改成 mouth_open3_dataset 所在位置
DATASET_ROOT = Path(r"C:\Users\user\Desktop\M2\Computer Vision and Image Processing\Major project\mouth_open3\mouth_open3\mouth_open3_dataset")

LABEL_NAMES = ["closed", "small", "wide"]
LABEL_TO_ID = {n: i for i, n in enumerate(LABEL_NAMES)}
# 供 VRM 驱动使用：类别 → 嘴巴张开强度
OPEN_AMOUNT = {"closed": 0.0, "small": 0.4, "wide": 1.0}

4.10.0


In [4]:
# 读取所有 labels.csv；人物编号直接取文件夹名（去掉最后的 session id）
records = []
for csv_path in sorted(DATASET_ROOT.rglob("labels.csv")):
    person = csv_path.parent.name.rsplit("_", 1)[0]
    with open(csv_path, newline="", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            records.append({
                "image_path": csv_path.parent / row["filename"],
                "person": person,
                "frame_index": int(row["frame_index"]),
                "label": row["label"],
                "y": LABEL_TO_ID[row["label"]],
            })

people = sorted({r["person"] for r in records})
print("人数：", len(people), "样本数：", len(records))
for p in people:
    c = Counter(r["label"] for r in records if r["person"] == p)
    print(f"{p:45s}", {n: c[n] for n in LABEL_NAMES})

人数： 11 样本数： 3432
8724461-hd_720_1366_25fps                     {'closed': 97, 'small': 354, 'wide': 24}
Elon Musk                                     {'closed': 166, 'small': 54, 'wide': 0}
Ishihara Satomi                               {'closed': 109, 'small': 102, 'wide': 0}
Michelle C. Smith                             {'closed': 87, 'small': 309, 'wide': 31}
Rachels English                               {'closed': 188, 'small': 328, 'wide': 30}
San Diego Voice and Accent                    {'closed': 72, 'small': 296, 'wide': 63}
train01_asian_female_12s                      {'closed': 33, 'small': 236, 'wide': 0}
train02_middleeastern_female_hijab_10s        {'closed': 26, 'small': 257, 'wide': 0}
train04_white_female_10s                      {'closed': 79, 'small': 160, 'wide': 0}
train06a_asian_male_6s                        {'closed': 27, 'small': 83, 'wide': 0}
train08_black_male_suit_10s                   {'closed': 19, 'small': 202, 'wide': 0}


In [5]:
# 和原来 step2.1 相同的 HOG 参数；CLAHE 可选
MOUTH_SIZE = (64, 32)   # (宽, 高)
HOG_PARAMS = {"_winSize": MOUTH_SIZE, "_blockSize": (16, 16), "_blockStride": (8, 8),
              "_cellSize": (8, 8), "_nbins": 9}
hog = cv2.HOGDescriptor(**HOG_PARAMS)
USE_CLAHE = False   # 实验里 CLAHE 没有提升（macro-F1 0.672 → 0.631），默认关闭
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))

def mouth_feature(gray):
    """灰度嘴部裁剪图 → 固定尺寸 → (CLAHE) → HOG 向量。视频预测时也调用这个函数。"""
    img = cv2.resize(gray, MOUTH_SIZE, interpolation=cv2.INTER_AREA)
    if USE_CLAHE:
        img = clahe.apply(img)
    return hog.compute(img).ravel()

def read_gray(path):
    return cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_GRAYSCALE)

X = np.stack([mouth_feature(read_gray(r["image_path"])) for r in records]).astype(np.float32)
y = np.array([r["y"] for r in records])
P = np.array([r["person"] for r in records])
F = np.array([r["frame_index"] for r in records])
print("特征矩阵：", X.shape)

特征矩阵： (3432, 756)


In [6]:
def smooth_labels(pred, frames, k=5):
    """按帧顺序做滑动窗口多数投票，去掉单帧跳变。"""
    order = np.argsort(frames); p = pred[order]; out = p.copy()
    for i in range(len(p)):
        out[i] = Counter(p[max(0, i - k // 2): i + k // 2 + 1]).most_common(1)[0][0]
    res = np.empty_like(out); res[order] = out
    return res

def leave_one_person_out(params):
    y_true, y_pred, y_smooth = [], [], []
    for person in people:
        tr, te = P != person, P == person
        model = SVC(gamma="scale", **params).fit(X[tr], y[tr])
        pred = model.predict(X[te])
        y_true.append(y[te]); y_pred.append(pred); y_smooth.append(smooth_labels(pred, F[te]))
    return np.concatenate(y_true), np.concatenate(y_pred), np.concatenate(y_smooth)

results = []
for kernel, C, cw in itertools.product(["linear", "rbf"], [0.1, 1.0, 10.0], [None, "balanced"]):
    params = {"kernel": kernel, "C": C, "class_weight": cw}
    t, p, s = leave_one_person_out(params)
    results.append({**params, "acc": accuracy_score(t, p), "macro_f1": f1_score(t, p, average="macro"),
                    "macro_f1_smooth": f1_score(t, s, average="macro")})
    print(f"{kernel:6s} C={C:<4} weight={str(cw):8s} | acc={results[-1]['acc']:.3f} macro-F1={results[-1]['macro_f1']:.3f}")

best = max(results, key=lambda r: r["macro_f1"])
best_params = {k: best[k] for k in ("kernel", "C", "class_weight")}
print("\n最佳参数：", best_params)

linear C=0.1  weight=None     | acc=0.771 macro-F1=0.604
linear C=0.1  weight=balanced | acc=0.712 macro-F1=0.585
linear C=1.0  weight=None     | acc=0.746 macro-F1=0.626
linear C=1.0  weight=balanced | acc=0.720 macro-F1=0.593
linear C=10.0 weight=None     | acc=0.725 macro-F1=0.587
linear C=10.0 weight=balanced | acc=0.731 macro-F1=0.601
rbf    C=0.1  weight=None     | acc=0.733 macro-F1=0.424
rbf    C=0.1  weight=balanced | acc=0.764 macro-F1=0.625
rbf    C=1.0  weight=None     | acc=0.791 macro-F1=0.626
rbf    C=1.0  weight=balanced | acc=0.766 macro-F1=0.619
rbf    C=10.0 weight=None     | acc=0.777 macro-F1=0.642
rbf    C=10.0 weight=balanced | acc=0.769 macro-F1=0.618

最佳参数： {'kernel': 'rbf', 'C': 10.0, 'class_weight': None}


In [7]:
# 最佳参数的留一人结果（这就是报告里用的数字）
t, p, s = leave_one_person_out(best_params)
baseline = accuracy_score(t, np.full_like(t, Counter(t).most_common(1)[0][0]))
print("全部猜最多类别的准确率：", round(baseline, 3))
print("\n逐帧预测：")
print(classification_report(t, p, target_names=LABEL_NAMES, digits=3))
print("加 5 帧平滑后：")
print(classification_report(t, s, target_names=LABEL_NAMES, digits=3))
print("混淆矩阵（行=真实，列=预测）", LABEL_NAMES)
print(confusion_matrix(t, s))

全部猜最多类别的准确率： 0.694

逐帧预测：
              precision    recall  f1-score   support

      closed      0.644     0.672     0.658       903
       small      0.845     0.841     0.843      2381
        wide      0.479     0.385     0.427       148

    accuracy                          0.777      3432
   macro avg      0.656     0.633     0.642      3432
weighted avg      0.776     0.777     0.776      3432

加 5 帧平滑后：
              precision    recall  f1-score   support

      closed      0.630     0.651     0.641       903
       small      0.839     0.845     0.842      2381
        wide      0.495     0.338     0.402       148

    accuracy                          0.772      3432
   macro avg      0.655     0.611     0.628      3432
weighted avg      0.769     0.772     0.770      3432

混淆矩阵（行=真实，列=预测） ['closed', 'small', 'wide']
[[ 588  314    1]
 [ 319 2012   50]
 [  26   72   50]]


In [8]:
# 用全部数据训练最终模型并保存，格式与原来的 pkl 一致，多了 open_amount
final_model = SVC(gamma="scale", **best_params).fit(X, y)
run_dir = DATASET_ROOT.parent / "mouth_open3_training_runs" / (datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6])
run_dir.mkdir(parents=True)
with open(run_dir / "mouth_open3_hog_svm.pkl", "wb") as f:
    pickle.dump({"model": final_model, "hog_params": HOG_PARAMS, "mouth_size": MOUTH_SIZE,
                 "use_clahe": USE_CLAHE, "label_names": LABEL_NAMES, "label_to_id": LABEL_TO_ID,
                 "open_amount": OPEN_AMOUNT, "best_params": best_params,
                 "preprocessing": "grayscale, resize 64x32, HOG" + (", CLAHE" if USE_CLAHE else "")}, f)
with open(run_dir / "loso_results.json", "w", encoding="utf-8") as f:
    json.dump({"search": results, "best_params": best_params, "baseline_acc": baseline,
               "report": classification_report(t, p, target_names=LABEL_NAMES, output_dict=True),
               "report_smooth": classification_report(t, s, target_names=LABEL_NAMES, output_dict=True),
               "confusion_smooth": confusion_matrix(t, s).tolist()}, f, indent=2, default=str)
print("已保存：", run_dir)

已保存： C:\Users\user\Desktop\M2\Computer Vision and Image Processing\Major project\mouth_open3\mouth_open3\mouth_open3_training_runs\20261029_024102_031478
